## WonderBox 포함 분석 Ver.3 : 년평균 평점-매출 데이터 --> WonderBox 컬렉션 합치기 --> Contribution --> 증감패턴  

## Main Collection 기준으로 집계 : 2025년 5월 7일

In [21]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [23]:
#df = pd.read_csv('review-vc_sales-by_collection_category_20250508_year.csv')
df = pd.read_csv('review-vc_sales-by_collection_category_20250508_year_v2.csv')

In [25]:
df['main_collection'].nunique()

612

In [27]:
df1 = df[['yr', 'financial_category', 'main_collection', 'written_avg_rating', 'sales_amount']]
df1 = df1.rename(columns={'financial_category':'category'})
df1 = df1.rename(columns={'main_collection':'collection1'})
df1 = df1.rename(columns={'written_avg_rating':'rating'})
df1 = df1.rename(columns={'sales_amount':'sales'})

In [29]:
df1 = df1[df1['collection1'] != '__TOTAL__']

In [31]:
df1[df1['yr']==2025]['sales'].sum()

99162610.35

In [291]:
df1

,yr,category,collection1,rating,sales
1,2022,Box Springs,Walter 9in,4.239130,1603178.28
2,2022,Box Springs,Walter 7.5in,4.207407,2758324.52
3,2022,Box Springs,Walter 4in,4.473333,2974705.85
4,2022,Box Springs,Victor 9in,4.621622,961075.44
5,2022,Box Springs,Victor 7.5in,4.717391,632951.63
...,...,...,...,...,...
2561,2025,Toppers,1.5in GT MF (+ GTFT),2.000000,9832.99
2562,2025,Toppers,1.5in GT MF (+ GTFT),2.000000,27214.48
2563,2025,Toppers,1.5in Copper Convoluted,NaN,NaN
2564,2025,Toppers,1.25in Swirl Copper,1.000000,136.05


In [293]:
agg_df = df1.groupby(['yr','category','collection1']).agg({
    'rating': 'mean',
    'sales': 'sum'
}).reset_index()

In [295]:
agg_df['collection1'].nunique()

611

In [297]:
df1[df1['yr']==2025]['sales'].sum()

99162610.35

In [299]:
#agg_df[agg_df['collection1']=='4in MyGel']
#agg_df[agg_df['collection1'].str.contains('MyGel', na=False)]

In [301]:
# yr 값을 컬럼으로 변환 (pivot)
pivot_df = agg_df.pivot_table(index=['category','collection1'],
                              columns='yr',
                              values=['rating','sales'])

In [303]:
# 컬럼 이름을 정리 (multi-index 해제)
#pivot_df.columns = [f"{val}_{yr}" for val, yr in pivot_df.columns]
pivot_df.columns = [f"{yr}_{val}" for val, yr in pivot_df.columns]

In [305]:
pivot_df.reset_index()

,category,collection1,2022_rating,2023_rating,2024_rating,2025_rating,2022_sales,2023_sales,2024_sales,2025_sales
0,Box Springs,5in OOT Box Spring,NaN,NaN,2.950000,3.194444,NaN,NaN,5258.64,240492.13
1,Box Springs,9in OOT Box Spring,NaN,NaN,3.242424,3.933333,NaN,NaN,7379.26,289665.80
2,Box Springs,Adrianne,NaN,NaN,NaN,NaN,913.93,0.00,0.00,0.00
3,Box Springs,Annemarie,4.186170,4.319767,4.341121,4.166667,2839354.78,2311639.79,2599136.15,455359.55
4,Box Springs,Armita 5in,4.099029,3.837037,3.346939,3.666667,11109930.21,5701328.78,410591.11,10242.95
...,...,...,...,...,...,...,...,...,...,...
607,Toppers,4in MyGel (+ MGT),4.075472,3.482759,3.541667,3.583333,436691.26,292672.49,203213.91,32995.61
608,Toppers,4in PRMF,NaN,NaN,NaN,NaN,14280.09,15607.23,120.99,0.00
609,Toppers,4in PRMF w Cover,5.000000,NaN,NaN,NaN,0.00,0.00,0.00,0.00
610,Toppers,4in Swirl Gel (+ SWFT),3.500000,3.666667,3.800000,1.428571,269305.47,153721.07,174365.99,33932.17


In [307]:
#pivot_df.to_csv('tmp_wonderbox_yearly_04.csv')

## 증감 패턴 구하기

In [311]:
#df = pd.read_csv('review_sales_wonderbox_out_02.csv')

In [313]:
pivot_df['2022_weight'] = pivot_df['2022_rating'] * pivot_df['2022_sales']
pivot_df['2023_weight'] = pivot_df['2023_rating'] * pivot_df['2023_sales']
pivot_df['2024_weight'] = pivot_df['2024_rating'] * pivot_df['2024_sales']
pivot_df['2025_weight'] = pivot_df['2025_rating'] * pivot_df['2025_sales']

In [315]:
y2022_total_weight = pivot_df['2022_weight'].sum()
pivot_df['2022_contribution'] = pivot_df['2022_weight']/y2022_total_weight

y2023_total_weight = pivot_df['2023_weight'].sum()
pivot_df['2023_contribution'] = pivot_df['2023_weight']/y2023_total_weight

y2024_total_weight = pivot_df['2024_weight'].sum()
pivot_df['2024_contribution'] = pivot_df['2024_weight']/y2024_total_weight

y2025_total_weight = pivot_df['2025_weight'].sum()
pivot_df['2025_contribution'] = pivot_df['2025_weight']/y2025_total_weight

In [317]:
pivot_df.head(5)

2022_rating  2023_rating  2024_rating  \
category    collection1                                                 
Box Springs 5in OOT Box Spring          NaN          NaN     2.950000   
            9in OOT Box Spring          NaN          NaN     3.242424   
            Adrianne                    NaN          NaN          NaN   
            Annemarie              4.186170     4.319767     4.341121   
            Armita 5in             4.099029     3.837037     3.346939   

                                2025_rating   2022_sales  2023_sales  \
category    collection1                                                
Box Springs 5in OOT Box Spring     3.194444          NaN         NaN   
            9in OOT Box Spring     3.933333          NaN         NaN   
            Adrianne                    NaN       913.93        0.00   
            Annemarie              4.166667   2839354.78  2311639.79   
            Armita 5in             3.666667  11109930.21  5701328.78   

                                2024_sales  2025_sales   2022_weight  \
category    collection1                                                
Box Springs 5in OOT Box Spring     5258.64   240492.13           NaN   
            9in OOT Box Spring     7379.26   289665.80           NaN   
            Adrianne                  0.00        0.00           NaN   
            Annemarie           2599136.15   455359.55  1.188602e+07   
            Armita 5in           410591.11    10242.95  4.553993e+07   

                                 2023_weight   2024_weight   2025_weight  \
category    collection1                                                    
Box Springs 5in OOT Box Spring           NaN  1.551299e+04  7.682387e+05   
            9in OOT Box Spring           NaN  2.392669e+04  1.139352e+06   
            Adrianne                     NaN           NaN           NaN   
            Annemarie           9.985746e+06  1.128317e+07  1.897331e+06   
            Armita 5in          2.187621e+07  1.374223e+06  3.755748e+04   

                                2022_contribution  2023_contribution  \
category    collection1                                                
Box Springs 5in OOT Box Spring                NaN                NaN   
            9in OOT Box Spring                NaN                NaN   
            Adrianne                          NaN                NaN   
            Annemarie                    0.004663           0.004951   
            Armita 5in                   0.017865           0.010846   

                                2024_contribution  2025_contribution  
category    collection1                                               
Box Springs 5in OOT Box Spring           0.000012           0.002339  
            9in OOT Box Spring           0.000019           0.003469  
            Adrianne                          NaN                NaN  
            Annemarie                    0.009051           0.005778  
            Armita 5in                   0.001102           0.000114

## Rating, Contribution 증감 패턴 구하기

In [320]:
# 1. rating_slope 컬럼 추가: 결측치 제외 후 연간 rating으로 기울기 계산
def calc_rating_slope(row):
    ratings = [row['2022_rating'], row['2023_rating'], row['2024_rating'], row['2025_rating']]
    years = [2022, 2023, 2024, 2025]

    # 결측치 아닌 값만 골라내기
    valid_x = []
    valid_y = []

    for i, r in enumerate(ratings):
        if pd.notna(r): 
            valid_x.append(i)
            valid_y.append(r)

    if len(valid_x) <= 1:
        return 0

    x = np.array(valid_x).reshape(-1,1)
    y = np.array(valid_y)

    model = LinearRegression()
    model.fit(x,y)
    return model.coef_[0]

In [322]:
pivot_df['rating_slope'] = pivot_df.apply(calc_rating_slope, axis=1)

In [324]:
def determine_trend(slope):
    if slope > 0: 
        return 1
    elif slope < 0:
        return -1
    else: 
        return 0

In [326]:
pivot_df['rating_trend'] = pivot_df['rating_slope'].apply(determine_trend)

In [328]:
#pivot_df[pivot_df['collection1'].str.contains('Armita')
pivot_df.head(5)

2022_rating  2023_rating  2024_rating  \
category    collection1                                                 
Box Springs 5in OOT Box Spring          NaN          NaN     2.950000   
            9in OOT Box Spring          NaN          NaN     3.242424   
            Adrianne                    NaN          NaN          NaN   
            Annemarie              4.186170     4.319767     4.341121   
            Armita 5in             4.099029     3.837037     3.346939   

                                2025_rating   2022_sales  2023_sales  \
category    collection1                                                
Box Springs 5in OOT Box Spring     3.194444          NaN         NaN   
            9in OOT Box Spring     3.933333          NaN         NaN   
            Adrianne                    NaN       913.93        0.00   
            Annemarie              4.166667   2839354.78  2311639.79   
            Armita 5in             3.666667  11109930.21  5701328.78   

                                2024_sales  2025_sales   2022_weight  \
category    collection1                                                
Box Springs 5in OOT Box Spring     5258.64   240492.13           NaN   
            9in OOT Box Spring     7379.26   289665.80           NaN   
            Adrianne                  0.00        0.00           NaN   
            Annemarie           2599136.15   455359.55  1.188602e+07   
            Armita 5in           410591.11    10242.95  4.553993e+07   

                                 2023_weight   2024_weight   2025_weight  \
category    collection1                                                    
Box Springs 5in OOT Box Spring           NaN  1.551299e+04  7.682387e+05   
            9in OOT Box Spring           NaN  2.392669e+04  1.139352e+06   
            Adrianne                     NaN           NaN           NaN   
            Annemarie           9.985746e+06  1.128317e+07  1.897331e+06   
            Armita 5in          2.187621e+07  1.374223e+06  3.755748e+04   

                                2022_contribution  2023_contribution  \
category    collection1                                                
Box Springs 5in OOT Box Spring                NaN                NaN   
            9in OOT Box Spring                NaN                NaN   
            Adrianne                          NaN                NaN   
            Annemarie                    0.004663           0.004951   
            Armita 5in                   0.017865           0.010846   

                                2024_contribution  2025_contribution  \
category    collection1                                                
Box Springs 5in OOT Box Spring           0.000012           0.002339   
            9in OOT Box Spring           0.000019           0.003469   
            Adrianne                          NaN                NaN   
            Annemarie                    0.009051           0.005778   
            Armita 5in                   0.001102           0.000114   

                                rating_slope  rating_trend  
category    collection1                                     
Box Springs 5in OOT Box Spring      0.244444             1  
            9in OOT Box Spring      0.690909             1  
            Adrianne                0.000000             0  
            Annemarie              -0.003716            -1  
            Armita 5in             -0.178719            -1

In [330]:
# 2. contribution_slope 컬럼 추가: 결측치 제외 후 연간 contribution 기울기 계산
def calc_contribution_slope(row):
    contributions = [row['2022_contribution'], row['2023_contribution'], row['2024_contribution'], row['2025_contribution']]
    years = [2022, 2023, 2024, 2025]

    # 결측치 아닌 값만 골라내기
    valid_x = []
    valid_y = []

    for i, r in enumerate(contributions):
        if pd.notna(r): 
            valid_x.append(i)
            valid_y.append(r)

    if len(valid_x) <= 1:
        return 0

    x = np.array(valid_x).reshape(-1,1)
    y = np.array(valid_y)

    model = LinearRegression()
    model.fit(x,y)
    return model.coef_[0]


In [332]:
pivot_df['contribution_slope'] = pivot_df.apply(calc_contribution_slope, axis=1)

In [333]:
pivot_df['contribution_trend'] = pivot_df['contribution_slope'].apply(determine_trend)

In [336]:
pivot_df

2022_rating  2023_rating  2024_rating  \
category    collection1                                                     
Box Springs 5in OOT Box Spring              NaN          NaN     2.950000   
            9in OOT Box Spring              NaN          NaN     3.242424   
            Adrianne                        NaN          NaN          NaN   
            Annemarie                  4.186170     4.319767     4.341121   
            Armita 5in                 4.099029     3.837037     3.346939   
...                                         ...          ...          ...   
Toppers     4in MyGel (+ MGT)          4.075472     3.482759     3.541667   
            4in PRMF                        NaN          NaN          NaN   
            4in PRMF w Cover           5.000000          NaN          NaN   
            4in Swirl Gel (+ SWFT)     3.500000     3.666667     3.800000   
            4in TorsoTec Copper             NaN          NaN          NaN   

                                    2025_rating   2022_sales  2023_sales  \
category    collection1                                                    
Box Springs 5in OOT Box Spring         3.194444          NaN         NaN   
            9in OOT Box Spring         3.933333          NaN         NaN   
            Adrianne                        NaN       913.93        0.00   
            Annemarie                  4.166667   2839354.78  2311639.79   
            Armita 5in                 3.666667  11109930.21  5701328.78   
...                                         ...          ...         ...   
Toppers     4in MyGel (+ MGT)          3.583333    436691.26   292672.49   
            4in PRMF                        NaN     14280.09    15607.23   
            4in PRMF w Cover                NaN         0.00        0.00   
            4in Swirl Gel (+ SWFT)     1.428571    269305.47   153721.07   
            4in TorsoTec Copper        3.000000         0.00        0.00   

                                    2024_sales  2025_sales   2022_weight  \
category    collection1                                                    
Box Springs 5in OOT Box Spring         5258.64   240492.13           NaN   
            9in OOT Box Spring         7379.26   289665.80           NaN   
            Adrianne                      0.00        0.00           NaN   
            Annemarie               2599136.15   455359.55  1.188602e+07   
            Armita 5in               410591.11    10242.95  4.553993e+07   
...                                        ...         ...           ...   
Toppers     4in MyGel (+ MGT)        203213.91    32995.61  1.779723e+06   
            4in PRMF                    120.99        0.00           NaN   
            4in PRMF w Cover              0.00        0.00  0.000000e+00   
            4in Swirl Gel (+ SWFT)   174365.99    33932.17  9.425691e+05   
            4in TorsoTec Copper           0.00        0.00           NaN   

                                     2023_weight   2024_weight   2025_weight  \
category    collection1                                                        
Box Springs 5in OOT Box Spring               NaN  1.551299e+04  7.682387e+05   
            9in OOT Box Spring               NaN  2.392669e+04  1.139352e+06   
            Adrianne                         NaN           NaN           NaN   
            Annemarie               9.985746e+06  1.128317e+07  1.897331e+06   
            Armita 5in              2.187621e+07  1.374223e+06  3.755748e+04   
...                                          ...           ...           ...   
Toppers     4in MyGel (+ MGT)       1.019308e+06  7.197159e+05  1.182343e+05   
            4in PRMF                         NaN           NaN           NaN   
            4in PRMF w Cover                 NaN           NaN           NaN   
            4in Swirl Gel (+ SWFT)  5.636439e+05  6.625908e+05  4.847453e+04   
            4in TorsoTec Copper              NaN           NaN  0.000000e+00   

               

In [338]:
def classify_pattern(row):
    if row['rating_trend'] > 0 and row['contribution_trend'] > 0:
        return 'P1'
    elif row['rating_trend'] > 0 and row['contribution_trend'] < 0:
        return 'P2'
    elif row['rating_trend'] < 0 and row['contribution_trend'] > 0:
        return 'P3'
    elif row['rating_trend'] < 0 and row['contribution_trend'] < 0:
        return 'P4'
    else:
        return 'P5'

In [340]:
pivot_df['pattern'] = pivot_df.apply(classify_pattern, axis=1)

In [342]:
pivot_df

2022_rating  2023_rating  2024_rating  \
category    collection1                                                     
Box Springs 5in OOT Box Spring              NaN          NaN     2.950000   
            9in OOT Box Spring              NaN          NaN     3.242424   
            Adrianne                        NaN          NaN          NaN   
            Annemarie                  4.186170     4.319767     4.341121   
            Armita 5in                 4.099029     3.837037     3.346939   
...                                         ...          ...          ...   
Toppers     4in MyGel (+ MGT)          4.075472     3.482759     3.541667   
            4in PRMF                        NaN          NaN          NaN   
            4in PRMF w Cover           5.000000          NaN          NaN   
            4in Swirl Gel (+ SWFT)     3.500000     3.666667     3.800000   
            4in TorsoTec Copper             NaN          NaN          NaN   

                                    2025_rating   2022_sales  2023_sales  \
category    collection1                                                    
Box Springs 5in OOT Box Spring         3.194444          NaN         NaN   
            9in OOT Box Spring         3.933333          NaN         NaN   
            Adrianne                        NaN       913.93        0.00   
            Annemarie                  4.166667   2839354.78  2311639.79   
            Armita 5in                 3.666667  11109930.21  5701328.78   
...                                         ...          ...         ...   
Toppers     4in MyGel (+ MGT)          3.583333    436691.26   292672.49   
            4in PRMF                        NaN     14280.09    15607.23   
            4in PRMF w Cover                NaN         0.00        0.00   
            4in Swirl Gel (+ SWFT)     1.428571    269305.47   153721.07   
            4in TorsoTec Copper        3.000000         0.00        0.00   

                                    2024_sales  2025_sales   2022_weight  \
category    collection1                                                    
Box Springs 5in OOT Box Spring         5258.64   240492.13           NaN   
            9in OOT Box Spring         7379.26   289665.80           NaN   
            Adrianne                      0.00        0.00           NaN   
            Annemarie               2599136.15   455359.55  1.188602e+07   
            Armita 5in               410591.11    10242.95  4.553993e+07   
...                                        ...         ...           ...   
Toppers     4in MyGel (+ MGT)        203213.91    32995.61  1.779723e+06   
            4in PRMF                    120.99        0.00           NaN   
            4in PRMF w Cover              0.00        0.00  0.000000e+00   
            4in Swirl Gel (+ SWFT)   174365.99    33932.17  9.425691e+05   
            4in TorsoTec Copper           0.00        0.00           NaN   

                                     2023_weight  ...   2025_weight  \
category    collection1                           ...                 
Box Springs 5in OOT Box Spring               NaN  ...  7.682387e+05   
            9in OOT Box Spring               NaN  ...  1.139352e+06   
            Adrianne                         NaN  ...           NaN   
            Annemarie               9.985746e+06  ...  1.897331e+06   
            Armita 5in              2.187621e+07  ...  3.755748e+04   
...                                          ...  ...           ...   
Toppers     4in MyGel (+ MGT)       1.019308e+06  ...  1.182343e+05   
            4in PRMF                         NaN  ...           NaN   
            4in PRMF w Cover                 NaN  ...           NaN   
            4in Swirl Gel (+ SWFT)  5.636439e+05  ...  4.847453e+04   
            4in TorsoTec Copper              NaN  ...  0.000000e+00   

                                    2022_contribution  2023_contribution  \
category    collection1                                 

In [344]:
pivot_df.to_csv('out_review_sales_wonderbox_pattern_yearly_v2_04.csv')